# Epic of Sun - Desafio 1
Benchmarking the models
***

This jupyter notebook benchmarks the models

It is divided in 2 parts:
* Loading libraries
* Benchmarking

***

## Loading libraries

In [35]:
import time
import numpy as np

import tensorflow as tf
from tensorflow.keras.models import load_model

from pathlib import Path
import json

## Benchmarking

In [36]:
# Get the directory where THIS script is located
base_path = Path.cwd()
base_path = base_path.parent  # Move one level up to the project root

In [37]:
# Function to load the model from the specified path

def load_model_from_path(model_path):
    if model_path.exists():
        try:
            # Pass str(model_path) to ensure compatibility
            model = load_model(str(model_path), compile=False)
            file_name = model_path.name
            print(f"Successfully loaded {file_name} model.")
            return model
        except Exception as e:
            print(f"File {file_name} exists, but load_model failed with error:\n{e}")
    else:
        print(f"File {file_name} does not exist at the specified path. Check your working directory or base_path.")

In [38]:
# Function to benchmark the model

def benchmark_model(model, input_shape=(1, 128, 128, 7), n_warmup=10, n_runs=100):
    """
    Benchmarks a trained segmentation model for inference cost.
    input_shape: (batch, H, W, C) -- use batch=1 to simulate single-patch
    edge inference, which is the realistic deployment scenario.
    """
    dummy_input = tf.random.uniform(input_shape)

    # Warm-up: excludes one-time graph tracing/compilation overhead from the timing
    for _ in range(n_warmup):
        _ = model(dummy_input, training=False)

    # Timed runs
    latencies = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model(dummy_input, training=False)
        latencies.append(time.perf_counter() - start)

    latencies = np.array(latencies) * 1000  # ms

    # Model size on disk (relevant for storage-constrained edge devices)
    n_params = model.count_params()
    param_bytes = sum(
        np.prod(w.shape) * np.dtype(str(w.dtype)).itemsize
        for w in model.trainable_weights
    )

    results = {
        "n_params": n_params,
        "param_size_mb": param_bytes / (1024 ** 2),
        "latency_mean_ms": latencies.mean(),
        "latency_p50_ms": np.percentile(latencies, 50),
        "latency_p95_ms": np.percentile(latencies, 95),
        "latency_std_ms": latencies.std(),
    }
    return results

In [39]:
# Function to print the results in a readable format
def print_benchmark_results(results, model_name="Model"):
    print(f"Benchmark results for {model_name}:")
    print(f"Number of parameters: {results['n_params']}")
    print(f"Parameter size (MB): {results['param_size_mb']:.2f}")
    print(f"Mean latency (ms): {results['latency_mean_ms']:.2f}")
    print(f"Median latency (ms): {results['latency_p50_ms']:.2f}")
    print(f"95th percentile latency (ms): {results['latency_p95_ms']:.2f}")
    print(f"Latency standard deviation (ms): {results['latency_std_ms']:.2f}")

    # Save benchmark data as a json file for future reference
    metrics_data = {
        "Model": model_name,
        "Number of parameters": results['n_params'],
        "Parameter size (MB)": results['param_size_mb'],
        "Mean latency (ms)": results['latency_mean_ms'],
        "Median latency (ms)": results['latency_p50_ms'],
        "95th percentile latency (ms)": results['latency_p95_ms'],
        "Latency standard deviation (ms)": results['latency_std_ms'],
    }

    metrics_path = base_path / "models" / "unet_benchmarks_data.json"
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    with open(str(metrics_path), "w") as f:
        json.dump(metrics_data, f)
    print(f"Benchmark data saved to '{metrics_path}'")

In [40]:
# Load the models

unet_model = load_model_from_path(base_path / "models" / "unet_sugarcane.keras")
#fpn_model = load_model_from_path(base_path / "models" / "fpn_sugarcane.keras")
#dsca_model = load_model_from_path(base_path / "models" / "dsca_pspnet_sugarcane.keras")

Successfully loaded unet_sugarcane.keras model.


In [41]:
# Usage per architecture:

results_unet = benchmark_model(unet_model)
#results_fpn = benchmark_model(fpn_model)
#results_dsca = benchmark_model(dsca_model)

In [42]:
# Print the results
print_benchmark_results(results_unet, model_name="U-Net")
#print_benchmark_results(results_fpn, model_name="FPN")
#print_benchmark_results(results_dsca, model_name="DSCA")

Benchmark results for U-Net:
Number of parameters: 7770081
Parameter size (MB): 29.62
Mean latency (ms): 49.55
Median latency (ms): 48.81
95th percentile latency (ms): 54.49
Latency standard deviation (ms): 2.39
Benchmark data saved to '/home/fdesmello/Git/spacesugar2/models/unet_benchmarks_data.json'
